# 第 1 章 · 核心组件（Core components）

> 对应文档：**Agents → Core components**
> https://docs.langchain.com/oss/python/langchain/agents#core-components
>
> 每个小节的结构都一样：**文档怎么说 → 平台上对应哪里 → 练习怎么做 → 看什么算证据**。
> Python 块是文档里的原写法，**只作对照，不用运行** —— 在这个平台上，同样的东西是
> 在 agent 编辑器的五个页签里填字段：Overview / Prompt / Tools / Skills / JSON。


## 文档的定义，先把这句话记住

> An agent is **a model calling tools in a loop** until a given task is complete.
> A harness is everything around that loop: the prompt, the tools, and any middleware
> that shapes the model's behavior.
> **Agent = Model + Harness.**

所以 agent 不是"更聪明的模型"，而是**模型 + 外面那圈东西**。这一章讲的就是那圈东西里最基础的五个：
`Model`、`Tools`、`System prompt`、`Structured output`、`Agent state`。

下面每个都对应编辑器里的一个位置，**没有一个是写代码才能设的**（第五个除外，见 5），
这就是这套材料和文档最大的差别：文档用 `create_agent(...)` 一次设完，平台用页签一次设一个。

## 1 · Model

> 文档：Pass a model identifier string (`"provider:model"`) or an initialized model
> instance to select the model for your agent.

```python
agent = create_agent(model="openai:gpt-5.5", tools=tools)
```

**平台上对应哪里**：agent 编辑器 → **Overview** → Model 下拉框（下面还有 Temperature 和 Max tokens）。

**练习**

1. Model 选 `deepseek-flash`，Temperature `0.2`，问：`给「静海城」写一句 8 个字的城市格言，只要格言本身。` 问三次。
2. 把 Temperature 改成 `1.2`，同样问三次。
3. Model 换成 `deepseek-v4-pro`，用同样的 prompt 再问一次。

**看什么算证据**

- 同一温度下三次是否一致：一致说明这个问题没有"多种合理答案"的空间，**不是温度没用**。
- 换模型后**格式/啰嗦程度**会变，事实不会变 —— 模型不带来事实。
- 想演示温度，问题必须选没有强关联答案的（格言可以，起名不行：三次都会叫"静海城"）。

## 2 · Tools

> Tools are what let the agent act. Each tool is a function with a name, a description,
> and a schema; the model chooses which to call and with what arguments.

```python
agent = create_agent(model="openai:gpt-5.5", tools=[search, calculator])
```

**平台上对应哪里**：agent 编辑器 → **Tools** → 勾选；右上角有 **[All tools]** / **[None]**。
面板上的 `N/M selected` 是唯一可信的数字。

**练习（这是本课最重要的一对）**

- **A 不给工具**：Tools 点 **[None]** → 保存 → 新聊天问 `1234567 × 891011 等于多少？`
- **B 给一个工具**：Tools 只勾 `python_execute` → 保存 → 新聊天，先问 `你好，介绍一下你自己。`
  再问上一题。

**看什么算证据**

- A：它照样给一个数字。**你无法核对它怎么算的。**
- B：第一句**不该**有工具调用；第二句回复下面应该出现一条工具调用记录，并在展开后看到它执行的代码。
- 判断"用了工具"只看**那条调用记录**，不看语气。答案的数字不重要，那次调用才是证据。

> 实测（2026-10-26，`deepseek-chat`）：A 给了数字但无法复现；B 的第一句 0 次调用、第二句
> 调了 `python_execute`，结果 `1100012777237` 可以自己核对。

## 3 · System prompt

> Shape how the agent approaches tasks. Accepts a string or `SystemMessage`.
> For dynamic prompts at runtime, use middleware.

```python
agent = create_agent(model="openai:gpt-5.5", tools=tools,
                     system_prompt="You are a helpful assistant. Be concise and accurate.")
```

**平台上对应哪里**：**Prompt** 页签。这里有两个东西，别混：

| 位置 | 进不进 prompt | 用途 |
|---|---|---|
| System Prompt | 每轮都进 | 角色、语气、输出形状 |
| Agent Docs → `AGENTS.md` | 每轮都进（内联嵌入） | 规则清单，长一点的东西放这里 |
| Agent Docs → `CLAUDE.md` | **不进** | 你自己的笔记 |

**练习**

1. System Prompt 用一句身份：`你是一位科幻小说作家，根据用户请求创建一个太空首都城市。`
   问：`月球的首都是哪里？`
2. 再改成"给两个例子"的写法（few-shot）：

```
你是一位科幻小说作家，根据用户请求创建一个太空首都城市。

用户：火星的首都是什么？
科幻作家：火星都 (Huǒxīng Dū)

用户：金星的首都是什么？
科幻作家：金星城 (Jīnxīng Chéng)
```

**看什么算证据**

- 第 1 步：它先澄清"月球没有首都"，然后**开始虚构** —— 一句身份就把事实题变成了创作题。
- 第 2 步：输出被"传染"成一行短名称。**例子比描述更彻底**：说清格式能拿到字段，给例子连长度都定了。
- 从第 1 步到第 2 步，模型没换、问题没变，**只改了几行字**。

## 4 · Structured output

> Return a validated schema from the agent using `response_format=`.

```python
from pydantic import BaseModel

class Answer(BaseModel):
    summary: str
    confidence: float

agent = create_agent(model="openai:gpt-5.5", tools=tools, response_format=Answer)
result = agent.invoke({"messages": [{"role": "user", "content": "Summarize AI trends"}]})
result["structured_response"]   # Answer(summary=..., confidence=...)
```

**平台上对应哪里**：**没有**。agent 配置里不存在 `response_format` 这一项。

**这不是不重要，而是这一版平台缺的一块**，所以要讲清它解决了什么：

- 返回值是**对象**，不是字符串 —— 程序可以直接读字段，不用正则去抠。
- 字段缺失/类型不对会**报错**，而不是悄悄给你一句"大概 3% 左右"。
- 对 agent 来说还有一层：它可以**先调工具、再交结构化结论**，让"说了什么"和"怎么算的"分开存。

**现在能做的替代**（都不完全等价）：

1. 在 System Prompt / `AGENTS.md` 里**规定字段**，要求按"名称 / 位置 / 氛围 / 经济"逐行输出；
2. 让 agent 用 `save_document` 把结果写进文件，文件有固定小标题 —— 人能读，程序仍然要解析。

**练习**：用第 1 种做法要求四行输出，再问一次事实题，看它是否**只**给四行。
实测：格式照做了，但前面仍有科普、后面仍有反问 —— **"说了字段"不等于"只有字段"**，
这正是 `response_format` 存在的理由。

## 5 · Agent state

> Every agent manages its execution context through `AgentState`, a typed dictionary that
> holds the current conversation history and any custom fields your tools and middleware need.

```python
from langchain.agents import AgentState, create_agent

class MyState(AgentState):
    user_id: str
    call_count: int

agent = create_agent(model="openai:gpt-5.5", tools=[], state_schema=MyState)
```

内置只有一个字段：`messages`（当前 thread 的全部对话历史，只追加不替换）。

**平台上对应哪里**：只能 JSON 页签（`backend: { type: "state", config: {}, mappings: {} }`），
没有可视化。

**但这件事本身要讲清楚，因为它一次解释了三个现象：**

1. **同一聊天里它记得上一轮** —— 那是 `messages` 在追加，不是"模型有记忆"。
2. **新聊天就忘了** —— 新 thread，`messages` 是空的。
3. **`AGENTS.md` / System Prompt 每轮都在** —— 它们在 `messages` 之外，属于 harness，不属于 state。

**练习**：同一聊天里先说 `我叫卫平，我喜欢蓝色`，再问 `我喜欢什么颜色？`；然后**新建聊天**问同一句。

**看什么算证据**

- 同一聊天：答得出。
- 新聊天：答不出。
- 结论：**"记住"是会话的属性，不是 agent 的能力。** 想让它不记得，只能换聊天；
  想让它跨聊天记得，需要 memory 中间件（见第 3 章）。

## 小结

| 文档 | 平台 | 状态 |
|---|---|---|
| Model | Overview → Model / Temperature / Max tokens | ✅ |
| Tools | Tools → 勾选 + `N/M selected` | ✅ |
| System prompt | Prompt → System Prompt + `AGENTS.md` | ✅（比文档多一层：规则和笔记分开） |
| Structured output | — | ❌ 缺 `response_format` |
| Agent state | JSON 页签 | ⚠️ 只能看，不能可视化设 |

下一章：**Invocation / Streaming** —— 同一段对话怎么被"跑起来"。